In [1]:
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

C:\Users\tripl\anaconda3\envs\deeplearning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("d0r1h/ILC")
train_set = pd.DataFrame(dataset['train'])
test_set = pd.DataFrame(dataset['test'])

In [3]:
train_set

,Title,Summary,Case
0,Company may continue arbitration despite appro...,The issue whether a company involved in arbitr...,IN THE CIVIL APPELLATE JURISDICTION CIVIL APPE...
1,"Arbitration, Conciliation and Mediation are th...",The objective of Arbitration is to settle the...,AFR Judgment reserved on 01.11.2021 Judgment ...
2,The petitioner was at best consuming narcotics...,It is directed that the petitioner be released...,IN THE HIGH COURT OF DELHI AT NEW DELHI NAMAN ...
3,Cases of Two Petitioners Identical Will Be Bar...,The two orders issued in exaltedly similarly q...,IN THE HIGH COURT OF JUDICATURE AT PATNA Civil...
4,Swachh Bharat Mission’s objective is to ensure...,The guidelines of Swachha Bharat Mission (Urba...,on 18 05 2021 on 22 03 civil ao st 9596 21 & ...
...,...,...,...
2053,No relaxation in the terms and conditions cont...,The selection process has to be conducted stri...,IN THE HIGH COURT OF JUDICATURE AT PATNA Civil...
2054,The intention of the parties must match the cl...,When the dispute revolves around the classific...,IN THE CIVIL APPELLATE JURISDICTION CIVIL APPE...
2055,High Court ought not to re-examine findings of...,Where a special statute governs the relationsh...,117IN THE HIGH COURT OF DELHI AT NEW DELHI De...
2056,The term ‘bail’ means a kind of security or bo...,Bail is the conditional release of a defendant...,Court No. 77 Case : CRIMINAL MISC. BAIL APPLIC...


In [4]:
test_set

,Title,Summary,Case
0,"S.86(1)(f) of the Electricity Act, is a specia...",Section 86(1)(f) vests a statutory jurisdictio...,Reportable IN THE CIVIL APPELLATE JURISDICTION...
1,"The petitioners were released on bail, as the ...",The petitioner apprehended arrest under Sectio...,IN THE HIGH COURT OF JUDICATURE AT PATNA CRIMI...
2,The allegation being only that the petitioner ...,The petitioner was arrested under Sections 344...,IN THE HIGH COURT OF JUDICATURE AT PATNA CRIMI...
3,"In service jurisprudence, seniority cannot be ...",In matters concerning administrative appointme...,IN THE HIGH COURT OF DELHI AT NEW DELHI Judgm...
4,The statute does not mandate all components of...,The facts and information of the suspected off...,on 06 04 2021 on 07 04 1606.19APPLN.odt1IN TH...
...,...,...,...
1010,Acceptance of public document could not be ref...,"In the present case, an appeal is preferred un...",IN THE HIGH COURT OF ORISSA AT CUTTACK RSA NO....
1011,Re-evaluation of answer sheets is not permissi...,Re-evaluation of answer sheets is not permissi...,IN THE HIGH COURT OF JHARKHAND AT RANCHI Kumar...
1012,Alternate accommodation for landlord alone can...,The presence of an alternate land that can be ...,IN THE HIGH COURT OF DELHI AT NEW DELHI Reser...
1013,Detention of applicant not necessary until cri...,Bail may be granted to an individual who is ac...,on 06 05 2021 on 06 05 rpa 1 5 43 ba 1355 202...


In [5]:
from rouge import Rouge
rouge = Rouge()

In [6]:

train_dataset = dataset['train'].select(range(300))
val_dataset = dataset['train'].select(range(50))

In [7]:
tokenizer = AutoTokenizer.from_pretrained("allenai/led-base-16384")
led = AutoModelForSeq2SeqLM.from_pretrained("allenai/led-base-16384", gradient_checkpointing=True, use_cache=False)

C:\Users\tripl\anaconda3\envs\deeplearning\lib\site-packages\transformers\modeling_utils.py:1038: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(resol

In [8]:
max_input_length = 8192
max_output_length = 600
batch_size = 2

In [9]:
def process_data_to_model_inputs(batch):

    inputs = tokenizer(
        batch["Case"],
        padding="max_length",
        truncation=True,
        max_length=max_input_length,
    )
    outputs = tokenizer(
        batch["Summary"],
        padding="max_length",
        truncation=True,
        max_length=max_output_length,
    )

    batch["input_ids"] = inputs.input_ids
    batch["attention_mask"] = inputs.attention_mask

    batch["global_attention_mask"] = len(batch["input_ids"]) * [
        [0 for _ in range(len(batch["input_ids"][0]))] ]
    batch["global_attention_mask"][0][0] = 1
    batch["labels"] = outputs.input_ids
    batch["labels"] = [
        [-100 if token == tokenizer.pad_token_id else token for token in labels]
        for labels in batch["labels"]]

    return batch

In [10]:
train_dataset = train_dataset.map(
    process_data_to_model_inputs,
    batched=True,
    batch_size=batch_size,
    remove_columns=['Title', 'Summary', 'Case'],
)

In [11]:
val_dataset = val_dataset.map(
    process_data_to_model_inputs,
    batched=True,
    batch_size=batch_size,
    remove_columns=['Title', 'Summary', 'Case'],
)

Map: 100%|█████████████████████████████████| 50/50 [00:00<00:00, 99.24 examples/s]


In [12]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "global_attention_mask", "labels"],
)
val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "global_attention_mask", "labels"],
)

In [13]:
led.config.num_beams = 2
led.config.max_length = 512
led.config.min_length = 100
led.config.length_penalty = 2.0
led.config.early_stopping = True
led.config.no_repeat_ngram_size = 3

In [14]:
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = tokenizer.pad_token_id
    label_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    rouge_output = rouge.compute(
        predictions=pred_str, references=label_str, rouge_types=["rouge2"]
    )["rouge2"].mid

    return {
        "rouge2_precision": round(rouge_output.precision, 4),
        "rouge2_recall": round(rouge_output.recall, 4),
        "rouge2_fmeasure": round(rouge_output.fmeasure, 4),
    }

In [15]:
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    evaluation_strategy="steps",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    fp16=False,
    output_dir="./",
    logging_steps=5,
    eval_steps=10,
    save_steps=10,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,

)

In [16]:
trainer = Seq2SeqTrainer(
            model=led,
            tokenizer=tokenizer,
            args=training_args,
            compute_metrics=compute_metrics,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
)

W&B installed but not logged in. Run `wandb login` or set the WANDB_API_KEY env variable.


In [ ]:
trainer.train()

C:\Users\tripl\anaconda3\envs\deeplearning\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


C:\Users\tripl\anaconda3\envs\deeplearning\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
